# Análisis exploratorio del Titanic

**Dataset:** Titanic — Kaggle  
**Archivo utilizado:** `train.csv`

## Situación

Se desea analizar la información disponible de los pasajeros del Titanic para identificar características asociadas con la supervivencia.

> **Importante:** En esta práctica no se realizará ningún modelo de Machine Learning. El trabajo se limita a limpieza, preprocesamiento, análisis exploratorio y visualización.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Carga del dataset

Se carga el archivo `dataset.csv`, correspondiente al conjunto de entrenamiento `train.csv` del Titanic.

In [ ]:
df = pd.read_csv("dataset.csv")

print(f"Número de pasajeros: {df.shape[0]}")
print(f"Número de columnas: {df.shape[1]}")

display(df.head())

## 2. Exploración inicial

Se revisan las variables disponibles, los tipos de datos, los valores faltantes, los registros duplicados y las estadísticas descriptivas.

In [ ]:
print("Variables disponibles:")
print(list(df.columns))

In [ ]:
print("Tipos de datos:")
display(df.dtypes.to_frame("Tipo de dato"))

In [ ]:
print("Información general:")
df.info()

In [ ]:
faltantes = pd.DataFrame({
    "Valores faltantes": df.isna().sum(),
    "Porcentaje (%)": df.isna().mean() * 100
}).sort_values("Valores faltantes", ascending=False)

display(faltantes)

In [ ]:
duplicados = df.duplicated().sum()
print(f"Registros duplicados: {duplicados}")

In [ ]:
print("Estadísticas descriptivas de variables numéricas:")
display(df.describe())

In [ ]:
print("Estadísticas descriptivas de variables categóricas:")
display(df.describe(include="object"))

## 3. Análisis de valores faltantes

Las variables `Age`, `Cabin` y `Embarked` requieren un tratamiento específico.

- **Age:** contiene valores faltantes. Se utilizará la mediana porque es una medida robusta frente a valores extremos y permite conservar los registros.
- **Cabin:** tiene una cantidad muy elevada de valores faltantes. En lugar de inventar números de cabina, se creará `HasCabin`, que indica si existe información de cabina. Después se eliminará `Cabin`.
- **Embarked:** tiene pocos valores faltantes. Al ser categórica, se completará utilizando la moda.

In [ ]:
df_clean = df.copy()

# Age: reemplazo de faltantes por la mediana
age_median = df_clean["Age"].median()
df_clean["Age"] = df_clean["Age"].fillna(age_median)

# Cabin: conservar únicamente si existe información
df_clean["HasCabin"] = df_clean["Cabin"].notna().astype(int)
df_clean.drop(columns=["Cabin"], inplace=True)

# Embarked: reemplazo de faltantes por la moda
embarked_mode = df_clean["Embarked"].mode()[0]
df_clean["Embarked"] = df_clean["Embarked"].fillna(embarked_mode)

print(f"Mediana utilizada para Age: {age_median:.2f}")
print(f"Moda utilizada para Embarked: {embarked_mode}")

print("\nValores faltantes después del tratamiento:")
display(df_clean.isna().sum().to_frame("Valores faltantes"))

## 4. Creación de nuevas variables

Se crearán cuatro variables nuevas:

### `FamilySize`
Representa el tamaño de la familia a bordo:

`FamilySize = SibSp + Parch + 1`

El `+1` corresponde al propio pasajero.

### `IsAlone`
Indica si el pasajero viajaba solo:

- `1`: viajaba solo.
- `0`: viajaba acompañado.

### `AgeGroup`
Se crean categorías de edad con estos criterios:

- **Niño:** menor de 13 años.
- **Joven:** de 13 a 17 años.
- **Adulto:** de 18 a 59 años.
- **Adulto mayor:** 60 años o más.

### `HasCabin`
Indica si el registro original contenía información de cabina:

- `1`: sí tenía información.
- `0`: no tenía información.

In [ ]:
df_clean["FamilySize"] = df_clean["SibSp"] + df_clean["Parch"] + 1
df_clean["IsAlone"] = (df_clean["FamilySize"] == 1).astype(int)

bins = [-np.inf, 12, 17, 59, np.inf]
labels = ["Niño", "Joven", "Adulto", "Adulto mayor"]

df_clean["AgeGroup"] = pd.cut(
    df_clean["Age"],
    bins=bins,
    labels=labels,
    right=True
)

display(df_clean.head())

## 5. Análisis 1 — ¿Qué porcentaje de pasajeros sobrevivió?

In [ ]:
total_pasajeros = len(df_clean)
sobrevivientes = df_clean["Survived"].sum()
no_sobrevivientes = total_pasajeros - sobrevivientes
porcentaje_supervivencia = df_clean["Survived"].mean() * 100

print(f"Pasajeros totales: {total_pasajeros:,}")
print(f"Sobrevivientes: {sobrevivientes:,}")
print(f"No sobrevivientes: {no_sobrevivientes:,}")
print(f"Porcentaje de supervivencia: {porcentaje_supervivencia:.2f}%")

## 6. Análisis 2 — ¿Cómo cambia la supervivencia entre hombres y mujeres?

In [ ]:
supervivencia_sexo = (
    df_clean.groupby("Sex")["Survived"]
    .mean()
    .mul(100)
    .rename("Supervivencia (%)")
    .reset_index()
)

display(supervivencia_sexo)

## 7. Análisis 3 — ¿Cómo cambia la supervivencia según la clase?

In [ ]:
supervivencia_clase = (
    df_clean.groupby("Pclass")["Survived"]
    .mean()
    .mul(100)
    .rename("Supervivencia (%)")
    .reset_index()
)

display(supervivencia_clase)

## 8. Análisis 4 — ¿Qué grupos de edad presentan mayor supervivencia?

In [ ]:
orden_edades = ["Niño", "Joven", "Adulto", "Adulto mayor"]

supervivencia_edad = (
    df_clean.groupby("AgeGroup", observed=False)["Survived"]
    .mean()
    .mul(100)
    .rename("Supervivencia (%)")
    .reindex(orden_edades)
    .reset_index()
)

display(supervivencia_edad)

## 9. Análisis adicional — ¿Viajar solo o acompañado parece estar relacionado con la supervivencia?

In [ ]:
supervivencia_compania = (
    df_clean.groupby("IsAlone")["Survived"]
    .mean()
    .mul(100)
    .rename("Supervivencia (%)")
    .reset_index()
)

supervivencia_compania["Tipo de viaje"] = supervivencia_compania["IsAlone"].map({
    0: "Acompañado",
    1: "Solo"
})

display(
    supervivencia_compania[["Tipo de viaje", "Supervivencia (%)"]]
)

## 10. Visualización 1 — Supervivencia general

Se muestra la cantidad de pasajeros que sobrevivieron y los que no sobrevivieron.

In [ ]:
plt.figure(figsize=(7, 5))

ax = sns.countplot(
    data=df_clean,
    x="Survived",
    order=[0, 1]
)

ax.set_title("Supervivencia de los pasajeros")
ax.set_xlabel("Supervivió")
ax.set_ylabel("Número de pasajeros")
ax.set_xticklabels(["No", "Sí"])

plt.tight_layout()
plt.show()

## 11. Visualización 2 — Supervivencia según sexo

In [ ]:
plt.figure(figsize=(7, 5))

ax = sns.barplot(
    data=supervivencia_sexo,
    x="Sex",
    y="Supervivencia (%)"
)

ax.set_title("Tasa de supervivencia según sexo")
ax.set_xlabel("Sexo")
ax.set_ylabel("Supervivencia (%)")
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

## 12. Visualización 3 — Supervivencia según clase

In [ ]:
plt.figure(figsize=(7, 5))

ax = sns.barplot(
    data=supervivencia_clase,
    x="Pclass",
    y="Supervivencia (%)"
)

ax.set_title("Tasa de supervivencia según clase")
ax.set_xlabel("Clase del pasajero")
ax.set_ylabel("Supervivencia (%)")
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

## 13. Visualización 4 — Supervivencia según grupo de edad

In [ ]:
plt.figure(figsize=(8, 5))

ax = sns.barplot(
    data=supervivencia_edad,
    x="AgeGroup",
    y="Supervivencia (%)",
    order=orden_edades
)

ax.set_title("Tasa de supervivencia según grupo de edad")
ax.set_xlabel("Grupo de edad")
ax.set_ylabel("Supervivencia (%)")
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

## 14. Conclusiones

A partir del análisis exploratorio realizado se pueden identificar diferencias en la supervivencia de los pasajeros según distintas características.

- La variable `Survived` permite calcular el porcentaje general de pasajeros que sobrevivieron.
- La supervivencia presenta diferencias entre los grupos de sexo.
- La clase del pasajero también presenta diferencias en las tasas de supervivencia.
- Los grupos de edad definidos muestran diferentes tasas de supervivencia.
- La variable `IsAlone` permite comparar descriptivamente a los pasajeros que viajaban solos con quienes viajaban acompañados.
- El tratamiento de valores faltantes permitió conservar los registros y mantener información útil: `Age` se completó con la mediana, `Embarked` con la moda y `Cabin` se transformó en `HasCabin`.

Los resultados son **descriptivos del conjunto `train.csv`**. Una diferencia entre grupos no demuestra por sí sola una relación causal.

**No se realizó ningún modelo de Machine Learning**, de acuerdo con las indicaciones de la práctica.

In [ ]:
# Comprobación final
print("Dataset original:", df.shape)
print("Dataset después del preprocesamiento:", df_clean.shape)

print("\nVariables nuevas:")
print(["FamilySize", "IsAlone", "AgeGroup", "HasCabin"])

print("\nValores faltantes restantes:")
display(df_clean.isna().sum().to_frame("Valores faltantes"))